In [1]:
import torch
import torch.nn as nn

In [2]:
from sklearn.preprocessing import StandardScaler

In [3]:
import pandas as pd

In [4]:
print(torch.cuda.is_available())
device= torch.device("cuda")

True


In [5]:
df=pd.read_csv(r"C:\Users\xhu70\Projects\twel_data_collection\data\sensor_data.csv")
df.head()

,timestamp,temperature_c,humidity_pct,pressure_hpa
0,2026-03-07T18:12:00,27.17,27.88,1012.39
1,2026-03-07T18:12:10,27.16,27.64,1012.42
2,2026-03-07T18:12:20,27.16,27.53,1012.41
3,2026-03-07T18:12:30,27.15,27.39,1012.39
4,2026-03-07T18:12:40,27.15,27.76,1012.44


In [6]:
features=["temperature_c","humidity_pct","pressure_hpa"]


scaler = StandardScaler()
X_np_raw= df[features].values
split_idx=int(len(X_np_raw)*0.8)
scaler = StandardScaler()
X_np = X_np_raw.copy()
X_np[:split_idx] = scaler.fit_transform(X_np_raw[:split_idx])  # 只看训练数据fit
X_np[split_idx:] = scaler.transform(X_np_raw[split_idx:])      # 测试集只transform

X_np

array([[ 0.43556875, -0.92089819, -0.36146916],
       [ 0.43356286, -0.93589927, -0.3546494 ],
       [ 0.43356286, -0.94277477, -0.35692265],
       ...,
       [-1.00666071,  1.53240443,  0.20684418],
       [-1.00666071,  1.52365379,  0.20457092],
       [-1.00866659,  1.52615397,  0.20457092]], shape=(59359, 3))

In [7]:
def make_windows(data, window_size):
    X,y=[],[]
    for i in range(len(data)-window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size,0])
    
    return torch.tensor(X,dtype=torch.float32), torch.tensor(y,dtype=torch.float32)

In [8]:
window_size=60  # 10min
X,y=make_windows(X_np, window_size)

C:\Users\xhu70\AppData\Local\Temp\ipykernel_51960\2961653156.py:7: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  return torch.tensor(X,dtype=torch.float32), torch.tensor(y,dtype=torch.float32)


In [9]:
split=int(len(X)*0.8)
X_train,X_test=X[:split],X[split:]
y_train,y_test=y[:split],y[split:]


In [10]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [11]:
import torch.nn as nn
import torch
import math
import torch.nn.functional as F

In [12]:
class TransformerForecast(nn.Module):
    def __init__(self, input_dim,d_model, num_heads, num_layers):
        super().__init__()
        self.embedding=nn.Linear(input_dim, d_model)
        self.d_model=d_model
        encoder_layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=num_heads, batch_first=True)
        self.transformer=nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.fc=nn.Linear(d_model,1)


    def get_positioning_encoding(self, seq_len, d_model):
        pe= torch.zeros(seq_len,d_model)
        position=torch.arange(0,seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * 
                            (-math.log(10000.0) / d_model))
        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)
        return pe.to(device)
    
    def forward(self, x):
        out=self.embedding(x)
        pe=self.get_positioning_encoding(len(x[0]),self.d_model)
        out=out+pe
        out=self.transformer(out)
        out=out[:,-1,:]
        out=self.fc(out)   #[32, 1]
        out=out.squeeze(-1)

        #print(out.shape)
        return out

    
model=TransformerForecast(input_dim=3, d_model=12, num_heads=4, num_layers=2).to(device)
batch_x,batch_y= next(iter(train_loader))
batch_x = batch_x.to(device) 
batch_y = batch_y.to(device) 

out= model(batch_x)

In [13]:
import torch.optim as optim
criterion=nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs=30

for epoch in range(num_epochs):
    model.train()
    total_loss=0

    for batch_x, batch_y in train_loader:
        batch_x=batch_x.to(device)
        batch_y=batch_y.to(device)
        optimizer.zero_grad()
        pred=model(batch_x)
        loss=criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    
    avg_loss= total_loss/ len(train_loader)
    print(epoch+1, avg_loss)

1 0.03684975254312987
2 0.010344304777484008
3 0.0060980872181856495
4 0.003968631979512944
5 0.0031627568736027445
6 0.0025197053155456735
7 0.0019002521551452485
8 0.0017801919803552427
9 0.0015428438888176169
10 0.0011780251190352863
11 0.001137244096304499
12 0.0008669787513036269
13 0.0006542237963557977
14 0.00056631212684876
15 0.00047870931931385434
16 0.0004633451814795276
17 0.00040506008037816013
18 0.00039407894548640376
19 0.00040244639596410974
20 0.000368787482510853
21 0.0003197567787373019
22 0.00033381289712153023
23 0.0003541464751783412
24 0.00032265074979101563
25 0.0002887179153563842
26 0.00028884986360275836
27 0.00026999830449839674
28 0.00027111676606923757
29 0.00028010007011014054
30 0.000244163796170146


In [14]:
model.eval()
preds, actuals=[],[]

with torch.no_grad():
    for batch_x,batch_y in test_loader:
        batch_x=batch_x.to(device)
        batch_y=batch_y.to(device)
        pred=model(batch_x)
        preds.append(pred.cpu())
        actuals.append(batch_y.cpu())

preds = torch.cat(preds)
actuals = torch.cat(actuals)

preds_np = preds.numpy().reshape(-1, 1)  # 变成[N, 1]

# 反归一化
# scaler是对3个特征fit的，温度是第0列
# 所以要构造一个假的3列数据，只有温度列有值

import numpy as np
dummy = np.zeros((len(preds_np), 3))
dummy[:, 0] = preds_np[:, 0]  # 温度放第0列
preds_real = scaler.inverse_transform(dummy)[:, 0]  # 只取温度列

# actuals同理
dummy2 = np.zeros((len(actuals), 3))
dummy2[:, 0] = actuals.numpy()
actuals_real = scaler.inverse_transform(dummy2)[:, 0]

# 现在算RMSE是真实摄氏度
rmse = np.sqrt(np.mean((preds_real - actuals_real) ** 2))
print(f"Test RMSE: {rmse:.4f} °C")

Test RMSE: 0.0417 °C


In [15]:
print(df['temperature_c'].min(), df['temperature_c'].max())
print(df['temperature_c'].std())

18.28 53.34
4.718797176290627
